# Log-Target + Remove Near-Constant Features

Train a simple `LGBMRegressor` on `log1p(target)` after removing features where the share of zeros is greater than `99%`.

In [1]:
import sys

sys.path.append("../")

import numpy as np
import pandas as pd

In [2]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error, root_mean_squared_log_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split

from src.loader import Loader
from src.preprocessing import FeaturePreprocessor

In [3]:
SEED = 42
TEST_SIZE = 0.33
CV = 5
ZERO_SHARE_THRESHOLD = 0.99

In [4]:
loader = Loader()
df = loader.load("../data/processed_data.csv")
df.shape

(4459, 4732)

In [5]:
X = df.drop(columns="target")
y = df["target"]
y_log = np.log1p(y)

(X.shape, y.shape)

((4459, 4731), (4459,))

In [6]:
X_train_full, X_test_full, y_train, y_test, y_train_log, y_test_log = train_test_split(
    X,
    y,
    y_log,
    test_size=TEST_SIZE,
    random_state=SEED,
)

preprocessor = FeaturePreprocessor(zero_share_threshold=ZERO_SHARE_THRESHOLD)
X_train = preprocessor.fit_transform(X_train_full)
X_test = preprocessor.transform(X_test_full)

pd.DataFrame(
    {
        "metric": ["features_before", "features_removed", "features_after"],
        "value": [X.shape[1], len(preprocessor.removed_sparse_columns_), X_train.shape[1]],
    }
)

,metric,value
0,features_before,4731
1,features_removed,2062
2,features_after,2669


In [7]:
cv = KFold(n_splits=CV, shuffle=True, random_state=SEED)

In [8]:
model = LGBMRegressor(
    random_state=SEED,
    n_jobs=-1,
    verbosity=-1,
)

In [9]:
# CV RMSE in log-space is equivalent to RMSLE for this setup.
cv_scores_rmsle = -cross_val_score(
    estimator=model,
    X=X_train,
    y=y_train_log,
    cv=cv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
)

pd.DataFrame(
    {
        "metric": ["cv_rmsle_mean", "cv_rmsle_std"],
        "value": [cv_scores_rmsle.mean(), cv_scores_rmsle.std()],
    }
)

,metric,value
0,cv_rmsle_mean,1.469250
1,cv_rmsle_std,0.034585


In [10]:
model.fit(X_train, y_train_log)

y_pred_log = model.predict(X_test)
y_pred = np.expm1(y_pred_log)
y_pred = np.clip(y_pred, 0, None)

In [11]:
metrics = pd.DataFrame(
    {
        "metric": ["rmsle", "rmse", "mae", "r2"],
        "value": [
            root_mean_squared_log_error(y_test, y_pred),
            root_mean_squared_error(y_test, y_pred),
            mean_absolute_error(y_test, y_pred),
            r2_score(y_test, y_pred),
        ],
    }
)

metrics.style.format({"value": "{:,.3f}"})

,metric,value
0,rmsle,1.483
1,rmse,"7,337,001.010"
2,mae,"4,173,037.371"
3,r2,0.157


## Conclusion

- Near-constant feature filtering is fit on the training split only, then applied to the test split, so the evaluation does not leak information from test into feature selection.
- The setup removes `2,062` near-constant sparse features and reduces the feature space from `4,731` to `2,669` columns.
- Since the task metric is `RMSLE`, the key CV result is `1.469250 +/- 0.034585`.
- On the held-out test split, the model gives `RMSLE = 1.483`, `RMSE = 7,337,001.010`, `MAE = 4,173,037.371`, and `R2 = 0.157`.
- Compared with the LightGBM baseline in `03_baseline.ipynb` (`CV RMSLE = 1.472 +/- 0.035`, `test RMSLE = 1.482`), removing near-constant features gives a very small CV improvement but does not improve held-out test `RMSLE`.
- Conclusion: combining `log1p(target)` with removal of features that have more than `99%` zeros is a reasonable experiment, but the current output does not make it clearly better than the simpler baseline from `03_baseline.ipynb`.


# 04 Remove Near-Constant Features Report

## Goal

The goal of this notebook was to test whether removing very sparse near-constant features improves the baseline model.

## What Was Done

- Loaded `data/processed_data.csv`.
- Split the data into train and test parts.
- Calculated zero share on the training split only.
- Removed features with more than `99%` zero values.
- Applied the selected feature set to both train and test data.
- Trained a LightGBM model with `log1p(target)`.
- Checked cross-validation and held-out test metrics.

## Feature Filtering Result

| metric | value |
| --- | ---: |
| Features before filtering | 4,731 |
| Features removed | 2,062 |
| Features after filtering | 2,669 |

## Model Results

| metric | value |
| --- | ---: |
| CV RMSLE mean | 1.469250 |
| CV RMSLE std | 0.034585 |
| Test RMSLE | 1.483 |
| Test RMSE | 7,337,001.010 |
| Test MAE | 4,173,037.371 |
| Test R2 | 0.157 |

## Conclusion

Near-constant sparse feature filtering produced a small CV improvement compared with the LightGBM baseline, but the held-out test RMSLE did not improve. This experiment is useful as a feature-filtering check, but it is not clearly better than the simpler baseline from notebook `03`.
